# MORPH Part 1: Boltz-2 structure prediction

This notebook validates a human UniProt accession, prepares a Boltz-2 input, runs the prediction, and downloads the resulting PDB.

Use a Google Colab GPU runtime. The output structure can be used as input to `2-MORPH.ipynb`.


#**⚠️ Before you start: enable a GPU runtime (Boltz-2 requires a GPU to run).**

In Google Colab:
1.   Click Runtime → Change runtime type
2.   Set Runtime type	Python 3
3.   Set Hardware accelerator to GPU (T4 recommended)
4.   Set Runtime version	2026.07
5.   **Click Save**







#**🤖 Let’s start by installing the required environment for Boltz-2**


In [ ]:

# @title 🧬 Install Boltz-2 v2.2.1 | Automatic Setup & Restart

import sys
import os
import signal
import time
import subprocess
import importlib.metadata as metadata

# ==================================================
# CONFIGURATION
# ==================================================
BOLTZ_VERSION = "2.2.1"
NUMPY_VERSION = "1.26.4"
EXPECTED_PYTHON = (3, 12)

print("=" * 60)
print("🧬 BOLTZ-2 AUTOMATED INSTALLER")
print(f"Boltz version: {BOLTZ_VERSION}")
print("Target: Google Colab 2026.07 | NVIDIA T4")
print("=" * 60, flush=True)

# ==================================================
# STEP 1: CHECK PYTHON VERSION
# ==================================================
print("\n[1/5] Checking Python environment...", flush=True)

python_version = sys.version_info[:2]
print("Python:", sys.version.split()[0])

if python_version != EXPECTED_PYTHON:
    raise RuntimeError(
        "❌ Incompatible Python version!\n"
        "Select Colab runtime version 2026.07 "
        "(Python 3.12) and reconnect."
    )

print("✅ Python version compatible.")

# ==================================================
# STEP 2: CHECK GPU
# ==================================================
print("\n[2/5] Checking NVIDIA GPU...", flush=True)

gpu_check = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name",
        "--format=csv,noheader"
    ],
    capture_output=True,
    text=True,
    check=False
)

if gpu_check.returncode != 0:
    raise RuntimeError(
        "❌ No NVIDIA GPU detected!\n"
        "Please select T4 GPU in Colab runtime settings."
    )

gpu_name = gpu_check.stdout.strip()
print("Detected GPU:", gpu_name)

if "T4" not in gpu_name:
    raise RuntimeError(
        f"❌ Expected NVIDIA T4, found: {gpu_name}\n"
        "Please select the T4 GPU runtime."
    )

print("✅ T4 GPU detected.")

# ==================================================
# STEP 3: INSTALL BOLTZ-2
# ==================================================
print("\n[3/5] Installing Boltz-2 and dependencies...")
print("⏳ This may take several minutes.\n", flush=True)

# Boltz is pinned to an official release.
# pip resolves the release's declared dependencies,
# including SciPy, Numba and PyTorch Lightning.
#
# NumPy is pinned to 1.26.4 because Boltz 2.2.1
# requires numpy>=1.26,<2.0.
#
# The optional CUDA-12 cuEquivariance packages are
# omitted for the older T4 GPU.
# PyTorch GPU acceleration remains available.

packages = [
    f"boltz=={BOLTZ_VERSION}",
    f"numpy=={NUMPY_VERSION}",
]

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--prefer-binary",
        "--upgrade-strategy", "only-if-needed",
        *packages
    ],
    check=True
)

print("\n✅ Package installation finished.", flush=True)

# ==================================================
# STEP 4: VERIFY INSTALLED DEPENDENCIES
# ==================================================
print("\n[4/5] Verifying installed versions...", flush=True)

required_packages = [
    "boltz",
    "numpy",
    "scipy",
    "numba",
    "torch",
    "pytorch-lightning"
]

for package in required_packages:
    version = metadata.version(package)
    print(f"  {package}: {version}")

if metadata.version("boltz") != BOLTZ_VERSION:
    raise RuntimeError("❌ Boltz version verification failed!")

if metadata.version("numpy") != NUMPY_VERSION:
    raise RuntimeError("❌ NumPy version verification failed!")

if metadata.version("scipy") != "1.13.1":
    raise RuntimeError("❌ SciPy version verification failed!")

if metadata.version("numba") != "0.61.0":
    raise RuntimeError("❌ Numba version verification failed!")

print("\n✅ Required package versions verified.")

# ==================================================
# STEP 5: AUTOMATIC RUNTIME RESTART
# ==================================================
print("\n[5/5] Restarting Python runtime...", flush=True)

print("\n" + "=" * 60)
print("✅ BOLTZ-2 INSTALLATION SUCCESSFUL!")
print("=" * 60)
print(f"🧬 Boltz version: {BOLTZ_VERSION}")
print(f"🎮 GPU: {gpu_name}")
print("📦 Dependencies installed and verified.")
print("")
print("🔄 Restarting Colab automatically...")
print("⏳ Please wait for the session to reconnect.")
print("▶️ Then continue to the next notebook cell.")
print("")
print("⚠️ A temporary disconnected/crashed message")
print("   during restart is expected.")
print("=" * 60, flush=True)

sys.stdout.flush()
sys.stderr.flush()
time.sleep(3)

# Force a Python kernel restart.
# Installed packages and selected runtime settings
# are normally retained.
os.kill(os.getpid(), signal.SIGKILL)


# **✅  Let's validate UniProt ID (human reference) and retrieve sequence**

In [ ]:
#@title Input UniProt ID, then run this cell
#@markdown This cell validates the UniProt entry and fetches the protein sequence.
#@markdown **Criteria (must pass):**
#@markdown - `UniProtKB reviewed (Swiss-Prot)`
#@markdown - `Homo sapiens (Human)`

uniprot_id = "Q9Y2D9"  #@param {type:"string"}

import requests

UNIPROT_ENTRY_URL = "https://rest.uniprot.org/uniprotkb/{accession}?format=json"

# Globals for downstream steps
current_uniprot_id = None
current_sequence = None

def fetch_uniprot_entry(accession: str) -> dict:
    accession = accession.strip()
    if not accession:
        raise ValueError("UniProt ID is empty.")
    url = UNIPROT_ENTRY_URL.format(accession=accession)
    r = requests.get(url)
    if r.status_code == 404:
        raise ValueError(f"No UniProtKB entry found for '{accession}'.")
    if not r.ok:
        raise ValueError(f"UniProt API error (HTTP {r.status_code}).")
    return r.json()

def validate_entry(entry: dict) -> tuple[bool, str]:
    entry_type = entry.get("entryType", "")
    sci_name = entry.get("organism", {}).get("scientificName", "")

    is_reviewed = entry_type == "UniProtKB reviewed (Swiss-Prot)"
    is_human = "Homo sapiens" in sci_name

    lines = []
    lines.append("✓ UniProtKB reviewed (Swiss-Prot)" if is_reviewed else f"✗ entryType: {entry_type}")
    lines.append("✓ Organism: Homo sapiens (Human)" if is_human else f"✗ Organism: {sci_name}")
    return (is_reviewed and is_human), "\n".join(lines)

def extract_sequence(entry: dict) -> str:
    seq = entry.get("sequence", {}).get("value")
    if not seq:
        raise ValueError("No sequence found in UniProt entry.")
    return seq.replace("\n", "").strip()

# ---- run ----
try:
    entry = fetch_uniprot_entry(uniprot_id)
    ok, msg = validate_entry(entry)

    print(f"Checking UniProt entry for {uniprot_id}\n")
    print(msg)
    print()

    if not ok:
        print("❌ Rejected: entry does not meet criteria.")
    else:
        seq = extract_sequence(entry)
        current_uniprot_id = uniprot_id.strip()
        current_sequence = seq
        print(f"✅ Accepted. Sequence length: {len(seq)} aa")
        # Optional: preview first 60 aa
        print("Sequence preview:", seq[:60] + ("..." if len(seq) > 60 else ""))

except Exception as e:
    print("❌ Error:", e)


# **🔧 Let's parameterize a Boltz-2 job**

In [ ]:
#@title Build Boltz-2 YAML (protein + ligands + constraints)
#@markdown Uses `current_sequence` from the UniProt step.
#@markdown - CCD ligands:   entries separated by ':' or ';' (optional explicit Chain-IDs via Chain-ID@CCD)
#@markdown - SMILES ligands: entries separated by ':' or ';' (NO explicit IDs; auto-assigned chain IDs)
#@markdown - Bond constraints (can provide multiple, separated by ':' or newlines):
#@markdown     CHAIN,RES,ATOM;CHAIN,RES,ATOM
#@markdown   Example:
#@markdown     A,461,SG;H,1,FE : A,462,SG;H,1,FE

jobname = "job1"  #@param {type:"string"}
protein_chain_id = "A"     #@param {type:"string"}

ligand_ccd = ""        #@param {type:"string"}
ligand_smiles = ""     #@param {type:"string"}

bond_constraints = ""      #@param {type:"string"}

print("Expected semantics:")
print("  - Protein: one entry with id=protein_chain_id and sequence=current_sequence")
print("  - Ligands:")
print("      * CCD   (ligand_ccd): 'HEM:HOH' or 'H@HEM;W@HOH' (':' or ';' separators)")
print("      * SMILES(ligand_smiles): 'CCO:CCN' or 'CCO;CCN'  (NO 'ID@...' allowed)")
print("        SMILES chains auto-assign after the protein chain (e.g., B, C, D...)")
print("  - Bond constraints (bond_constraints):")
print("      * One or more entries separated by ':' or newlines")
print("      * Each entry: CHAIN,RES,ATOM;CHAIN,RES,ATOM")
print("        Example: A,461,SG;H,1,FE")
print()

import re
import string
from pathlib import Path
import yaml

# --- YAML helpers to force inline [A, 461, SG] style ---
class FlowSeq(list):
    """A list that should be emitted in YAML flow style: [a, b, c]."""
    pass

class IndentDumper(yaml.SafeDumper):
    def increase_indent(self, flow=False, indentless=False):
        return super().increase_indent(flow, False)

def represent_flowseq(dumper, data):
    return dumper.represent_sequence("tag:yaml.org,2002:seq", data, flow_style=True)

IndentDumper.add_representer(FlowSeq, represent_flowseq)

# --- parsing helpers ---
def split_multi_sep(s: str):
    """Split on ':' or ';' for ligand fields."""
    s = (s or "").strip()
    if not s:
        return []
    return [x.strip() for x in re.split(r"[:;]", s) if x.strip()]

def split_bond_entries(s: str):
    """
    Split multiple bond constraints on ':' or newlines.
    NOTE: ';' is reserved inside each bond entry (atom1 ; atom2).
    """
    s = (s or "").strip()
    if not s:
        return []
    parts = []
    for line in s.splitlines():
        line = line.strip()
        if not line:
            continue
        parts.extend([x.strip() for x in line.split(":") if x.strip()])
    return parts

def parse_one_bond(entry: str):
    """
    Parse one bond entry: CHAIN,RES,ATOM;CHAIN,RES,ATOM
    Emits exactly:
      {"bond": {"atom1": [A, 461, SG], "atom2": [H, 1, FE]}}
    with atom1/atom2 lists in flow style.
    """
    entry = (entry or "").strip()
    if not entry:
        return None
    if ";" not in entry:
        raise ValueError("Each bond entry must use ';' between the two atoms: CHAIN,RES,ATOM;CHAIN,RES,ATOM")

    left, right = [x.strip() for x in entry.split(";", 1)]
    a1 = [x.strip() for x in left.split(",")]
    a2 = [x.strip() for x in right.split(",")]

    if len(a1) != 3 or len(a2) != 3:
        raise ValueError("Bond entry must be CHAIN,RES,ATOM;CHAIN,RES,ATOM (3 comma-separated fields each side)")

    try:
        res1 = int(a1[1])
        res2 = int(a2[1])
    except ValueError:
        raise ValueError("Bond entry residue indices (RES) must be integers.")

    return {
        "bond": {
            "atom1": FlowSeq([a1[0], res1, a1[2]]),
            "atom2": FlowSeq([a2[0], res2, a2[2]]),
        }
    }

def parse_ccd_with_optional_ids(raw: str):
    items = split_multi_sep(raw)
    out = []
    for it in items:
        if "@" in it:
            cid, val = it.split("@", 1)
            cid = cid.strip()
            val = val.strip()
            if not cid or not val:
                raise ValueError(f"Invalid CCD entry '{it}'. Use ID@CCD or CCD.")
            out.append((cid, val, True))
        else:
            out.append((None, it, False))
    return out

def parse_smiles_no_ids(raw: str):
    items = split_multi_sep(raw)
    for it in items:
        if "@" in it:
            raise ValueError(
                "SMILES input must NOT include explicit chain IDs. "
                "Remove 'ID@...' and provide only SMILES separated by ':' or ';'."
            )
    return items

def make_auto_chain_generator(start_after: str):
    letters = list(string.ascii_uppercase)
    start_idx = letters.index(start_after) + 1 if start_after in letters else 1
    return iter(letters[start_idx:])

# Require UniProt sequence from previous step
if "current_sequence" not in globals() or current_sequence is None:
    raise RuntimeError("Run the UniProt validation cell first (so current_sequence exists).")

seq_entries = [{"protein": {"id": protein_chain_id, "sequence": current_sequence}}]
auto_chain_iter = make_auto_chain_generator(protein_chain_id)

# CCD ligands (optional explicit IDs)
for cid, ccd, has_id in parse_ccd_with_optional_ids(ligand_ccd):
    chain = cid if has_id else next(auto_chain_iter)
    seq_entries.append({"ligand": {"id": chain, "ccd": ccd.upper()}})

# SMILES ligands (NO explicit IDs; always auto chain)
for smi in parse_smiles_no_ids(ligand_smiles):
    chain = next(auto_chain_iter)
    seq_entries.append({"ligand": {"id": chain, "smiles": smi}})

# Bond constraints (multiple entries allowed via ':' or newlines)
constraints = []
for entry in split_bond_entries(bond_constraints):
    b = parse_one_bond(entry)
    if b:
        constraints.append(b)

config = {"version": 1, "sequences": seq_entries}
if constraints:
    config["constraints"] = constraints

out_dir = Path("boltz_inputs")
out_dir.mkdir(exist_ok=True)
yaml_path = out_dir / f"{jobname}.yaml"

yaml_text = yaml.dump(
    config,
    Dumper=IndentDumper,
    sort_keys=False,
    default_flow_style=False,
    indent=2,
    width=10_000,
)

yaml_path.write_text(yaml_text)

print("✅ Wrote Boltz-2 YAML to:", yaml_path)
print("\n--- YAML preview ---\n")
print(yaml_text)


# **📥 Download generated Boltz-2 YAML (Optional)**




In [ ]:
#@title View the generated YAML (notepad-style)
#@markdown Shows and download the most recent YAML in `boltz_inputs/` in a scrollable "notepad" box.

from pathlib import Path
from IPython.display import display, HTML
import html
from google.colab import files

files.download(str(yaml_path))

yaml_dir = Path("boltz_inputs")
yaml_files = sorted(yaml_dir.glob("*.yaml"), key=lambda p: p.stat().st_mtime)

if not yaml_files:
    raise RuntimeError("No YAML files found in boltz_inputs/. Run the YAML generation cell first.")

yaml_path = yaml_files[-1]
yaml_text = yaml_path.read_text()

# escape for HTML display
safe_text = html.escape(yaml_text)

display(HTML(f"""
<div style="
  border: 1px solid #ddd;
  border-radius: 8px;
  padding: 12px;
  background: #fafafa;
  font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', monospace;
  font-size: 13px;
  line-height: 1.4;
  white-space: pre;
  overflow: auto;
  max-height: 420px;
">
<b>{yaml_path.name}</b>\n
{safe_text}
</div>
"""))


#**🚀  Run Boltz-2**

In [ ]:
#@title Run Boltz-2 prediction on most recently generated YAML (using MSA server). This can take a little while ... Stretch break 🤸‍♂️ 🧘‍♀️

import subprocess
from pathlib import Path

# Find most recently generated YAML
yaml_dir = Path("boltz_inputs")
yaml_files = sorted(
    yaml_dir.glob("*.yaml"),
    key=lambda p: p.stat().st_mtime
)

if not yaml_files:
    raise RuntimeError(
        "No YAML found. Generate the Boltz-2 YAML first."
    )

yaml_path = yaml_files[-1]
out_dir = Path("boltz_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

job_out_dir = out_dir / f"boltz_results_{yaml_path.stem}"
prediction_dir = job_out_dir / "predictions"

# Run prediction
cmd = [
    "boltz", "predict",
    str(yaml_path),
    "--use_msa_server",
    "--out_dir", str(out_dir),
    "--no_kernels",
    "--override"
]

print("🚀 Running Boltz-2")
print("YAML:", yaml_path)
print("GPU: NVIDIA T4")
print("Output:", job_out_dir)
print("\nCommand:", " ".join(cmd))
print("\n" + "=" * 60, flush=True)

# Show the full Boltz log instead of hiding it
result = subprocess.run(
    cmd,
    check=False
)

if result.returncode != 0:
    raise RuntimeError(
        f"Boltz-2 failed (exit code {result.returncode}). "
        "Check the log above."
    )

# Verify that actual structures were produced
cif_files = sorted(prediction_dir.rglob("*.cif"))

if not cif_files:
    raise RuntimeError(
        "Boltz exited without an error, but NO CIF "
        "structure was generated.\n"
        f"Expected predictions under: {prediction_dir}\n"
        "Check the Boltz log above for skipped "
        "predictions or inference errors."
    )

# Save best-ranked model for subsequent cells
preferred = [
    p for p in cif_files
    if p.name == f"{yaml_path.stem}_model_0.cif"
]

prediction_cif_path = (
    preferred[0] if preferred else cif_files[0]
)

print("\n" + "=" * 60)
print("✅ BOLTZ-2 PREDICTION SUCCESSFUL")
print("=" * 60)
print("Generated CIF files:", len(cif_files))
print("Selected structure:", prediction_cif_path)
print("\n✅ Ready for visualization!")


#**🔬 View prediction**

In [ ]:
#@title py3Dmol protein rendering (rainbow cartoon; HETATM as sticks.)


!pip -q install biopython py3Dmol

import py3Dmol
from pathlib import Path
from Bio.PDB import MMCIFParser, PDBIO

cif_path = Path("boltz_outputs/boltz_results_job1/predictions/job1/job1_model_0.cif")
pdb_path = cif_path.with_suffix(".pdb")

assert cif_path.exists(), f"Missing CIF file: {cif_path}"

# CIF → PDB
parser = MMCIFParser(QUIET=True)
structure = parser.get_structure("boltz_model", str(cif_path))

io = PDBIO()
io.set_structure(structure)
io.save(str(pdb_path))   # ← important fix

pdb_text = pdb_path.read_text()

view = py3Dmol.view(width=900, height=550)
view.addModel(pdb_text, "pdb")

# Clear any default styles
view.setStyle({}, {})

# Protein (ATOM) → rainbow cartoon
view.setStyle(
    {"hetflag": False},
    {"cartoon": {"color": "spectrum"}}
)

# HETATM → sticks
view.setStyle(
    {"hetflag": True},
    {"stick": {"radius": 0.25}}
)

view.zoomTo()
view.show()


#**📥 Download boltz-2 prediction as `UniProt-ID.pdb`**

In [ ]:
#@title Download .pdb

from pathlib import Path
import shutil

# pick the UniProt id you set earlier
uid = (globals().get("current_uniprot_id") or globals().get("uniprot_id") or "uniprotid").strip()

# this should already exist from your CIF→PDB step
src = globals().get("pdb_path", None)
if src is None:
    raise ValueError("pdb_path not found. Make sure you ran the CIF→PDB conversion cell first.")
src = Path(src)

if not src.exists():
    raise FileNotFoundError(f"Missing PDB file: {src}")

dst = Path(f"{uid}.pdb")
shutil.copyfile(src, dst)
print(f"✅ Saved as: {dst.resolve()}")

# Download (Colab) or show a link (Jupyter)
try:
    from google.colab import files  # type: ignore
    files.download(str(dst))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(dst)))
